# Goal-conditioned RL baselines

> MangoBench's decentralized goal-conditioned offline RL baselines (JAX): one agent per agent.

In [ ]:
#| default_exp wm.gcrl.core

In [ ]:
#| hide
from nbdev.showdoc import *

MangoBench's fully decentralized baselines: **one goal-conditioned agent per agent**, each trained
on its own observations, actions and goals only (MangoBench's `MultiAgentManager`). The agents are
the vendored JAX implementations (`stable_marl.wm.gcrl.ogcrl`, see its README):

| name here | MangoBench | algorithm |
|---|---|---|
| `gcbc` | GCMBC | goal-conditioned behaviour cloning |
| `crl` | ICRL | contrastive RL |
| `hiql` | IHIQL | hierarchical implicit Q-learning |
| `gcivl` | IGCIVL | goal-conditioned implicit V-learning |
| `gciql` | (not in the paper) | goal-conditioned implicit Q-learning |

In MangoBench, the "agents" are parts of one robot, and `batch_partition` cuts each agent's slice out
of the robot's state. MultiGrid agents have their own observations, so the agent axis of our datasets
plays that role: `agent_datasets` turns a dataset into one OGBench dataset per agent.

Goals are sampled from each agent's own trajectory (hindsight relabelling, by OGBench's `GCDataset`),
and at evaluation each agent is conditioned on its goal in the infos: `goal` for `pixels`,
``goal_<key>`` for the other observation keys (e.g. `goal_position`).

In [ ]:
#| export
from __future__ import annotations

import importlib
import pickle
from pathlib import Path
from typing import Any, Callable, Sequence

import numpy as np

### Per-agent datasets

Each agent's trajectory ends when it terminates (reaching its goal) or with the episode. OGBench's
compact format holds every state of a trajectory: ``terminals`` marks its last state, which is not a
transition (``valids`` = 0) but can be a goal.

In [ ]:
#| export
GCRL_AGENTS = ('gcbc', 'crl', 'hiql', 'gcivl', 'gciql')


def goal_key(obs_key: str) -> str:
    "Info key of the goal matching observation key `obs_key`: ``goal`` for ``pixels``, else ``goal_<key>``."
    return 'goal' if obs_key == 'pixels' else f'goal_{obs_key}'


def _obs_keys(obs_keys: str | Sequence[str]) -> list[str]:
    keys = [obs_keys] if isinstance(obs_keys, str) else list(obs_keys)
    if 'pixels' in keys and len(keys) > 1:
        raise ValueError("observations are either the agents' `pixels` or feature keys (e.g. ['position']), not both")
    return keys


def _agent_obs(columns: dict[str, np.ndarray], keys: list[str]) -> np.ndarray:
    "Observations (N, ...) of one agent: its images (uint8), or its feature keys concatenated (float32)."
    if keys == ['pixels']:
        return np.asarray(columns['pixels'], np.uint8)
    n = len(columns[keys[0]])
    return np.concatenate([np.asarray(columns[k], np.float32).reshape(n, -1) for k in keys], axis=1)


def agent_datasets(dataset, obs_keys: str | Sequence[str] = 'pixels') -> list[dict[str, np.ndarray]]:
    """
    One OGBench dataset (compact format: ``observations``, ``actions``, ``terminals``, ``valids``) per
    agent of `dataset` (episodes of ``(steps, num_agents, ...)`` columns, with ``action`` and
    ``terminated``). Agent ``a``'s trajectory in an episode runs from the reset to the step where it
    terminated (its goal), or to the end of the episode.
    """
    keys = _obs_keys(obs_keys)
    cols = {k: dataset.get_col_data(k) for k in (*keys, 'action', 'terminated')}
    num_agents = cols['action'].shape[1]
    out = []
    for a in range(num_agents):
        rows = []
        for start, length in zip(dataset.offsets, dataset.lengths):
            done = np.flatnonzero(cols['terminated'][start:start + length, a])
            end = int(done[0]) if len(done) else int(length) - 1   # last state of the trajectory
            rows.append(np.arange(start, start + end + 1))
        lengths = [len(r) for r in rows]
        rows = np.concatenate(rows)
        terminals = np.zeros(len(rows), np.float32)
        terminals[np.cumsum(lengths) - 1] = 1.0
        actions = np.nan_to_num(cols['action'][rows, a], nan=0.0)   # the last state's action is unused
        actions = np.where(terminals > 0, 0, actions).astype(np.int32)
        obs = _agent_obs({k: cols[k][rows, a] for k in keys}, keys)
        out.append(dict(observations=obs, actions=actions, terminals=terminals, valids=1.0 - terminals))
    return out

### Configurations

`gcrl_config` starts from the agent's OGBench configuration with OGBench's settings for discrete actions
and image observations (its Powderworld tasks: batches of 256, AWR actors with `alpha=3`, HIQL with
`subgoal_steps=10` and `low_actor_rep_grad`), then applies `overrides` (e.g. MangoBench's
`alpha=0.1` for CRL on AntMaze).

In [ ]:
#| export
_DISCRETE_DEFAULTS = {
    'gcbc': dict(batch_size=256),
    'gcivl': dict(batch_size=256, alpha=3.0),
    'gciql': dict(batch_size=256, alpha=3.0, actor_loss='awr'),
    'crl': dict(batch_size=256, alpha=3.0, actor_loss='awr'),
    'hiql': dict(batch_size=256, high_alpha=3.0, low_alpha=3.0, low_actor_rep_grad=True, subgoal_steps=10),
}


def _agent_module(name: str):
    if name not in GCRL_AGENTS:
        raise ValueError(f"unknown agent {name!r}: one of {GCRL_AGENTS}")
    return importlib.import_module(f'stable_marl.wm.gcrl.ogcrl.agents.{name}')


def gcrl_config(agent_name: str, images: bool = True, **overrides):
    """
    The ``ml_collections.ConfigDict`` of `agent_name` for discrete actions: OGBench's defaults, its
    discrete-action settings, the ``impala_small`` encoder if `images`, then `overrides`.
    """
    config = _agent_module(agent_name).get_config()
    config.discrete = True
    if images:
        config.encoder = 'impala_small'
    for key, value in {**_DISCRETE_DEFAULTS[agent_name], **overrides}.items():
        if key not in config:
            raise KeyError(f"{agent_name} has no config key {key!r}")
        config[key] = value
    return config

### Independent agents

`IndependentGCRL` holds one agent per MultiGrid agent. It trains them on their own datasets (each
agent samples its own batches) and acts as the model of a :class:`FeedForwardPolicy`: ``get_action``
reads each agent's latest observation and its goal from the infos.

In [ ]:
#| export
def _gc_dataset(data: dict, config):
    from stable_marl.wm.gcrl.ogcrl.utils.datasets import Dataset, GCDataset, HGCDataset
    cls = {'GCDataset': GCDataset, 'HGCDataset': HGCDataset}[config['dataset_class']]
    return cls(Dataset.create(**{k: np.array(v) for k, v in data.items()}), config)


class IndependentGCRL:
    """
    One goal-conditioned agent (`agent_name` in :data:`GCRL_AGENTS`) per agent, trained independently.

    Parameters
    ----------
    agents : list
        The agents (vendored OGBench agents), one per MultiGrid agent
    agent_name : str
        Their algorithm
    config : ml_collections.ConfigDict
        Their configuration (see :func:`gcrl_config`)
    obs_keys : str or list of str
        Info keys of the observations: ``'pixels'`` (each agent's view), or feature keys (e.g.
        ``['position']``); each needs a goal (see :func:`goal_key`)
    num_actions : int
        Number of discrete actions
    temperature : float
        Action sampling temperature when acting (0: the most likely action)
    seed : int
        Seed of the action sampling
    """
    def __init__(self, agents: list, agent_name: str, config, obs_keys: str | Sequence[str], num_actions: int,
                 temperature: float = 0.0, seed: int = 0):
        import jax
        self.agents, self.agent_name, self.config = list(agents), agent_name, config
        self.obs_keys, self.num_actions, self.temperature = _obs_keys(obs_keys), num_actions, temperature
        self._rng = jax.random.PRNGKey(seed)
        self._examples: list[np.ndarray] = []   # one observation per agent, to rebuild the agents

    @property
    def num_agents(self) -> int:
        return len(self.agents)

    @classmethod
    def create(cls, agent_name: str, agent_data: list[dict], num_actions: int, obs_keys: str | Sequence[str] = 'pixels',
               seed: int = 0, temperature: float = 0.0, **overrides) -> IndependentGCRL:
        """
        Agents for the per-agent datasets `agent_data` (see :func:`agent_datasets`), agent ``i`` seeded
        with ``seed + i``; `overrides` go to :func:`gcrl_config`.
        """
        keys = _obs_keys(obs_keys)
        config = gcrl_config(agent_name, images=keys == ['pixels'], **overrides)
        examples = [data['observations'][:1] for data in agent_data]
        model = cls([], agent_name, config, keys, num_actions, temperature, seed)
        model.agents = model._build(examples, seed)
        return model

    def _build(self, example_observations: list[np.ndarray], seed: int) -> list:
        from stable_marl.wm.gcrl.ogcrl.agents import agents
        self._examples = [np.zeros_like(ex) for ex in example_observations]
        ex_actions = np.full((1,), self.num_actions - 1, np.int32)   # discrete: the largest action
        return [agents[self.agent_name].create(seed + i, ex, ex_actions, self.config)
                for i, ex in enumerate(example_observations)]

    def update(self, batches: list[dict]) -> list[dict]:
        "One gradient step of each agent on its batch; their update infos."
        infos = []
        for i, batch in enumerate(batches):
            self.agents[i], info = self.agents[i].update(batch)
            infos.append(info)
        return infos

    def train(self, agent_data: list[dict], steps: int, log_every: int = 1000, seed: int | None = None,
              log: Callable[[dict], None] | None = None) -> dict[str, list]:
        """
        `steps` updates of every agent on batches of its dataset (``config.batch_size``). Every
        `log_every` steps, each agent's losses are averaged into ``history`` (and passed to `log`).
        OGBench's datasets sample with numpy's global generator: `seed` seeds it.
        """
        if len(agent_data) != self.num_agents:
            raise ValueError(f"{len(agent_data)} datasets for {self.num_agents} agents")
        if seed is not None:
            np.random.seed(seed)
        datasets = [_gc_dataset(data, self.config) for data in agent_data]
        history: dict[str, list] = {'step': []}
        sums: dict[str, float] = {}
        for step in range(1, steps + 1):
            infos = self.update([ds.sample(self.config['batch_size']) for ds in datasets])
            for i, info in enumerate(infos):
                for k, v in info.items():
                    sums[f'agent{i}/{k}'] = sums.get(f'agent{i}/{k}', 0.0) + float(v)
            if step % log_every == 0 or step == steps:
                n = log_every if step % log_every == 0 else step % log_every
                row = {'step': step, **{k: v / n for k, v in sums.items()}}
                for k, v in row.items():
                    history.setdefault(k, []).append(v)
                sums = {}
                if log: log(row)
        return history

    # --- acting, as the model of a FeedForwardPolicy ---
    def eval(self) -> IndependentGCRL:
        return self

    def parameters(self):
        return iter(())

    def _inputs(self, info: dict, keys: list[str], a: int) -> np.ndarray:
        "Agent `a`'s inputs at the latest step, (num_envs, ...), prepared as in :func:`agent_datasets`."
        cols = []
        for k in keys:
            if k not in info:
                raise KeyError(f"{k!r} is not in the infos (keys: {sorted(info)})")
            v = info[k]
            cols.append((v.cpu().numpy() if hasattr(v, 'cpu') else np.asarray(v))[:, -1, a])
        if self.obs_keys == ['pixels']:
            return np.asarray(cols[0], np.uint8)
        return np.concatenate([np.asarray(c, np.float32).reshape(len(c), -1) for c in cols], axis=1)

    def get_action(self, info: dict) -> np.ndarray:
        "Actions (num_envs, num_agents) for stacked infos ``(num_envs, time, num_agents, ...)``."
        import jax
        goal_keys = [goal_key(k) for k in self.obs_keys]
        actions = []
        for a, agent in enumerate(self.agents):
            self._rng, key = jax.random.split(self._rng)
            act = agent.sample_actions(observations=self._inputs(info, self.obs_keys, a),
                                       goals=self._inputs(info, goal_keys, a), seed=key, temperature=self.temperature)
            actions.append(np.asarray(act))
        return np.stack(actions, axis=1).astype(np.int64)

    # --- checkpoints ---
    def save(self, path: str | Path) -> Path:
        "Save the agents' parameters and how to rebuild them to `path` (a pickle)."
        import flax
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        state = dict(agent_name=self.agent_name, config=self.config.to_dict(), obs_keys=self.obs_keys,
                     num_actions=self.num_actions, temperature=self.temperature,
                     examples=self._examples, agents=[flax.serialization.to_state_dict(ag) for ag in self.agents])
        with open(path, 'wb') as f:
            pickle.dump(state, f)
        return path

    @classmethod
    def load(cls, path: str | Path, seed: int = 0) -> IndependentGCRL:
        import flax, ml_collections
        with open(path, 'rb') as f:
            state = pickle.load(f)
        config = ml_collections.ConfigDict(state['config'])
        model = cls([], state['agent_name'], config, state['obs_keys'], state['num_actions'], state['temperature'], seed)
        model.agents = [flax.serialization.from_state_dict(ag, s)
                        for ag, s in zip(model._build(state['examples'], seed), state['agents'])]
        return model

### Tests

In [ ]:
import tempfile
from fastcore.test import test_fail
from stable_marl import World
from stable_marl.policy import FeedForwardPolicy, RandomPolicy
from stable_marl.envs.multigrid import GoToGoalPolicy
from stable_marl.data import HDF5Dataset

tmp = tempfile.mkdtemp()
kw = dict(agents=2, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=4, max_steps=15)
world = World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, **kw)
world.set_policy(GoToGoalPolicy())
world.collect(f'{tmp}/fg.h5', episodes=6, seed=0, progress=False)
world.set_policy(RandomPolicy(seed=0))
world.collect(f'{tmp}/fg_random.h5', episodes=4, seed=100, progress=False)
ds = HDF5Dataset(path=f'{tmp}/fg.h5')

# one compact dataset per agent: its trajectory stops where it reached its goal
data = agent_datasets(ds)
term = ds.get_col_data('terminated')
assert len(data) == 2
for a, d in enumerate(data):
    assert d['observations'].shape[1:] == (28, 28, 3) and d['observations'].dtype == np.uint8
    assert d['terminals'].sum() == len(ds.lengths) and (d['valids'] == 1 - d['terminals']).all()
    ends = np.flatnonzero(d['terminals'])
    starts = np.r_[0, ends[:-1] + 1]
    for e, (s, t) in enumerate(zip(starts, ends)):
        rows = slice(ds.offsets[e], ds.offsets[e] + ds.lengths[e])
        reached = np.flatnonzero(term[rows, a])
        assert t - s == (reached[0] if len(reached) else ds.lengths[e] - 1)          # the expert reaches its goal
        assert np.array_equal(d['observations'][s], ds.get_col_data('pixels')[ds.offsets[e], a])
feats = agent_datasets(ds, ['position'])
assert feats[0]['observations'].shape[1:] == (2,) and feats[0]['observations'].dtype == np.float32
test_fail(lambda: agent_datasets(ds, ['pixels', 'position']), contains='not both')
test_fail(lambda: gcrl_config('qrl'), contains='unknown agent')
test_fail(lambda: gcrl_config('gcbc', nope=1), contains='nope')
assert gcrl_config('crl', images=False, alpha=0.1).alpha == 0.1 and gcrl_config('hiql').encoder == 'impala_small'

# every agent trains, acts in a World through a FeedForwardPolicy, and saves / loads
num_actions = world.envs.envs[0].unwrapped.action_space[0].n
for name in GCRL_AGENTS:
    model = IndependentGCRL.create(name, data, num_actions, seed=0, batch_size=16, **({'subgoal_steps': 3} if name == 'hiql' else {}))
    hist = model.train(data, steps=4, log_every=2, seed=0)
    assert hist['step'] == [2, 4] and any(k.startswith('agent1/') for k in hist), name
    world.set_policy(FeedForwardPolicy(model))
    res = world.evaluate(episodes=2, seed=0)
    assert len(res['episode_successes']) == 2
    world.reset(seed=3)
    actions = model.get_action(world.infos)
    assert actions.shape == (2, 2) and ((0 <= actions) & (actions < num_actions)).all()
    loaded = IndependentGCRL.load(model.save(f'{tmp}/{name}.pkl'))
    assert np.array_equal(loaded.get_action(world.infos), actions), name               # temperature 0: deterministic

# feature observations: conditioned on goal_position
model = IndependentGCRL.create('gcivl', feats, num_actions, obs_keys=['position'], batch_size=16)
model.train(feats, steps=2)
world.reset(seed=3)
assert model.get_action(world.infos).shape == (2, 2)
info = {k: v for k, v in world.infos.items() if k != 'goal_position'}
test_fail(lambda: model.get_action(info), contains='goal_position')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()